# House prices from Land Registry Price Paid → ITL2

Builds **median house price per ITL2 region per year, with new-build/existing and property-type splits**, from raw HM Land Registry Price Paid transactions. Saves `clean/house_prices_ppd_itl2.csv`.

- **Source:** HM Land Registry **Price Paid Data**, per-year bulk CSVs (`pp-YYYY.csv`) — one row per residential sale in **England & Wales** since 1995.
- **Why raw transactions:** we take the **median of all sales** in a region-year, which is well-defined and robust to luxury-sale outliers — no weighting/median-of-medians problem.
- **Two inputs needed:** (1) the `pp-YYYY.csv` files in `raw/ppd/`; (2) the **ONS Postcode Directory (ONSPD)** in `raw/onspd/`, because Price Paid locates sales by **postcode** — we map postcode → local authority → (crosswalk) → ITL2.
- **Coverage:** England & Wales only (Scotland/NI have separate registers) — the 7 Scottish/NI regions come out blank, same gap as migration.
- **Cleaning:** keep PPD **category A** (standard sales) only; residential types Detached/Semi/Terraced/Flat.

Edit only `BASE` and (optionally) the `YEARS` range.

In [1]:
import re
from pathlib import Path
import pandas as pd

BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")

PPD_DIR   = BASE / "raw" / "housing" / "price_paid"    # pp-YYYY.csv files here
ONSPD_DIR = BASE / "raw" / "onspd"      # ONS Postcode Directory csv here
SPINE     = BASE / "clean" / "geography_spine.csv"
CROSSWALK = BASE / "lib" / "la_code_crosswalk.csv"
OUT       = BASE / "clean" / "house_prices_ppd_itl2.csv"

YEARS = range(2018, 2025)               # panel window; widen if you want more history

# Price Paid files are HEADERLESS - these are the columns in order:
PP_COLS = ["txn_id","price","date","postcode","property_type","old_new","duration",
           "paon","saon","street","locality","town","district","county",
           "ppd_category","record_status"]
USE = ["price","date","postcode","property_type","old_new","ppd_category"]

def norm_pc(s):
    return s.astype(str).str.upper().str.replace(" ", "", regex=False)

## 1. Build the postcode → local authority lookup (from ONSPD)

Load only the postcode and LA-code columns from the ONS Postcode Directory, normalise the postcodes (upper-case, no spaces), and build a fast lookup.

In [2]:
onspd_files = list(ONSPD_DIR.rglob("ONSPD*.csv")) + list(ONSPD_DIR.rglob("*.csv"))
if not onspd_files:
    raise SystemExit(f"No ONSPD csv in {ONSPD_DIR}. Download the ONS Postcode Directory there.")
onspd_file = onspd_files[0]
print("ONSPD:", onspd_file.name)

# peek at header to find the postcode + LA-district columns (names vary: pcds/pcd, oslaua/laua/lad)
head = pd.read_csv(onspd_file, nrows=0)
cols = {c.lower(): c for c in head.columns}
pc_col  = cols.get("pcds") or cols.get("pcd") or cols.get("pcd2")
lad_col = cols.get("oslaua") or cols.get("laua") or cols.get("lad")
if not (pc_col and lad_col):
    raise SystemExit(f"Couldn't find postcode/LAD columns. Available: {list(head.columns)}")
print(f"Using postcode column '{pc_col}' and LA column '{lad_col}'")

pcl = pd.read_csv(onspd_file, usecols=[pc_col, lad_col], dtype=str)
pcl["pc"] = norm_pc(pcl[pc_col])
pc_to_lad = dict(zip(pcl["pc"], pcl[lad_col]))
print(f"Postcode lookup built: {len(pc_to_lad):,} postcodes")

ONSPD: ONSPD_FEB_2026_UK.csv


SystemExit: Couldn't find postcode/LAD columns. Available: ['pcd7', 'pcd8', 'pcds', 'dointr', 'doterm', 'cty25cd', 'ced25cd', 'lad25cd', 'wd25cd', 'parncp25cd', 'usrtypind', 'east1m', 'north1m', 'gridind', 'hlth19cd', 'nhser24cd', 'ctry25cd', 'rgn25cd', 'ssr95cd', 'pcon24cd', 'eer20cd', 'educ23cd', 'ttwa15cd', 'pco19cd', 'itl25cd', 'wdstl05cd', 'oa01cd', 'wdcas03cd', 'npark16cd', 'lsoa01cd', 'msoa01cd', 'ruc01ind', 'oac01ind', 'oa11cd', 'lsoa11cd', 'msoa11cd', 'wz11cd', 'sicbl24cd', 'bua24cd', 'ruc11ind', 'oac11ind', 'lat', 'long', 'lep21cd1', 'lep21cd2', 'pfa23cd', 'imd20ind', 'cal24cd', 'icb23cd', 'oa21cd', 'lsoa21cd', 'msoa21cd', 'ruc21ind']

/Users/h.cantekin/.conda/envs/charts/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 2. Build spine + crosswalk lookups

We map LA → ITL2 via the spine, applying the code crosswalk first so retired LA codes in the ONSPD resolve to current ones.

In [ ]:
spine = pd.read_csv(SPINE)
lad_to_itl2 = dict(zip(spine.la_code, spine.itl2_code))
itl2_name = dict(zip(spine.itl2_code, spine.itl2_name))

remap = {}
if CROSSWALK.exists():
    xw = pd.read_csv(CROSSWALK, dtype=str)
    remap = dict(zip(xw.old_code, xw.new_code))
print(f"Spine: {len(set(spine.itl2_code))} ITL2 regions | crosswalk: {len(remap)} recodes")

## 3. Process each year and aggregate to ITL2

For each `pp-YYYY.csv`: keep category-A residential sales, map postcode→LA→ITL2, then take medians (all sales, and the splits) per region. Processing one year at a time keeps memory manageable.

In [ ]:
RES = {"D","S","T","F"}          # residential property types
TYPE_NAME = {"D":"detached","S":"semi_detached","T":"terraced","F":"flat"}

def aggregate_year(df):
    g = df.groupby("itl2_code")
    out = g["price"].median().rename("median_all").to_frame()
    out["n_sales"] = g.size()
    # new-build vs existing
    piv_new = df.pivot_table(index="itl2_code", columns="old_new", values="price", aggfunc="median")
    out["median_newbuild"] = piv_new.get("Y")
    out["median_existing"] = piv_new.get("N")
    # by property type
    piv_t = df.pivot_table(index="itl2_code", columns="property_type", values="price", aggfunc="median")
    for code, name in TYPE_NAME.items():
        out[f"median_{name}"] = piv_t.get(code)
    return out.reset_index()

frames = []
for yr in YEARS:
    f = PPD_DIR / f"pp-{yr}.csv"
    if not f.exists():
        print(f"  {yr}: file not found ({f.name}) - skipped"); continue
    df = pd.read_csv(f, header=None, names=PP_COLS, usecols=USE, dtype=str)
    df = df[(df["ppd_category"] == "A") & (df["property_type"].isin(RES))]
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = df.dropna(subset=["price", "postcode"])
    # postcode -> LA -> (crosswalk) -> ITL2
    lad = norm_pc(df["postcode"]).map(pc_to_lad)
    lad = lad.replace(remap)
    df["itl2_code"] = lad.map(lad_to_itl2)
    n_unmapped = df["itl2_code"].isna().sum()
    df = df.dropna(subset=["itl2_code"])
    a = aggregate_year(df)
    a["year"] = yr
    frames.append(a)
    print(f"  {yr}: {len(df):>7,} sales -> {a['itl2_code'].nunique()} regions "
          f"({n_unmapped:,} unmapped postcodes dropped)")

panel = pd.concat(frames, ignore_index=True)
panel["itl2_name"] = panel["itl2_code"].map(itl2_name)
cols = ["itl2_code","itl2_name","year","median_all","median_newbuild","median_existing",
        "median_detached","median_semi_detached","median_terraced","median_flat","n_sales"]
panel = panel[cols].sort_values(["year","itl2_code"]).reset_index(drop=True)
OUT.parent.mkdir(parents=True, exist_ok=True)
panel.to_csv(OUT, index=False)
print(f"\nSaved -> {OUT}")
panel.head()

## 4. Coverage & sanity check

Expect ~39 of 46 regions (E&W only). Inner London should show the highest median prices; new-build vs existing and the property-type ordering (flats < terraced < semi < detached, roughly) are quick plausibility checks.

In [ ]:
regions = spine[["itl2_code","itl2_name"]].drop_duplicates()
covered = set(panel.itl2_code.unique())
missing = regions[~regions.itl2_code.isin(covered)]
latest = panel.year.max()
print(f"Coverage: {len(covered)} of {len(regions)} regions x {panel.year.nunique()} years ({panel.year.min()}-{latest})")
if len(missing):
    print(f"No data ({len(missing)} regions, expected E&W-only): " + ", ".join(missing.itl2_name.tolist()))
print(f"\n{latest} highest / lowest median_all:")
y = panel[panel.year == latest]
print(pd.concat([y.nlargest(3,"median_all"), y.nsmallest(3,"median_all")])
        [["itl2_code","itl2_name","median_all","median_newbuild","median_flat","n_sales"]].to_string(index=False))